In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import dataclass, asdict
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="muted")

# Loading Dataset

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print("Datasets loaded!")

# Exploratory Data Analysis

## Basic Dataset Statistics

In [ ]:
print("-" * 50)
print("BASIC DATASET STATISTICS")
print("-" * 50)
print(f"Total rows: {len(train)}")
print(f"Missing values:\n{train.isnull().sum()}\n")

## Checking Class Distribution for Imbalance

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=train, x='answer', order=['A', 'B', 'C', 'D', 'E'])
plt.title("Distribution of Correct Answers")
plt.xlabel("Options")
plt.ylabel("Count")
    
# Add percentage labels
total = len(train)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.show()

## Text Analysis

In [ ]:
print("-" * 50)
print("TEXT LENGTH ANALYSIS (Word Counts)")
print("-" * 50)

df = train.copy()

# Calculate word counts
df['prompt_len'] = df['prompt'].apply(lambda x: len(str(x).split()))
options = ['A', 'B', 'C', 'D', 'E']
for opt in options:
    df[f'{opt}_len'] = df[opt].apply(lambda x: len(str(x).split()))
        
# Calculate the max option length for each row
df['max_option_len'] = df[[f'{opt}_len' for opt in options]].max(axis=1)
    
# Total length = Prompt + Longest Option (This determines max_length for BERT)
df['total_max_len'] = df['prompt_len'] + df['max_option_len']
    
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
sns.histplot(df['prompt_len'], bins=30, ax=axes[0], color='skyblue', kde=True)
axes[0].set_title('Prompt Length Distribution')
    
sns.histplot(df['max_option_len'], bins=30, ax=axes[1], color='lightgreen', kde=True)
axes[1].set_title('Longest Option Length Distribution')
    
sns.histplot(df['total_max_len'], bins=30, ax=axes[2], color='salmon', kde=True)
axes[2].set_title('Total Sequence Length (Prompt + Max Option)')
axes[2].axvline(df['total_max_len'].quantile(0.99), color='red', linestyle='dashed', label='99th Percentile')
axes[2].legend()
    
plt.tight_layout()
plt.show()
    
print(f"Max Prompt Length: {df['prompt_len'].max()} words")
print(f"Max Option Length: {df['max_option_len'].max()} words")
print(f"Max Total Sequence: {df['total_max_len'].max()} words")
print(f"99th Percentile Total Sequence: {df['total_max_len'].quantile(0.99):.0f} words")

In [ ]:
print("-" * 50)
print("STRUCTURAL BIAS: Does length correlate with correctness?")
print("-" * 50)
    
# Find the length of the correct answer for each row
def get_correct_answer_len(row):
    correct_letter = row['answer']
    return row[f'{correct_letter}_len']
        
df['correct_answer_len'] = df.apply(get_correct_answer_len, axis=1)
    
# Calculate average lengths
avg_correct_len = df['correct_answer_len'].mean()
    
# Calculate average length of INCORRECT answers
def get_avg_incorrect_len(row):
    correct = row['answer']
    incorrect_lens = [row[f'{opt}_len'] for opt in options if opt != correct]
    return np.mean(incorrect_lens)
        
df['avg_incorrect_len'] = df.apply(get_avg_incorrect_len, axis=1)
avg_incorrect_len = df['avg_incorrect_len'].mean()
    
print(f"Average length of CORRECT answers: {avg_correct_len:.2f} words")
print(f"Average length of INCORRECT answers: {avg_incorrect_len:.2f} words")
    
if avg_correct_len > avg_incorrect_len * 1.1:
    print("BIAS DETECTED: Correct answers tend to be significantly longer than incorrect ones.")
else:
    print("NO OBVIOUS BIAS: Correct and incorrect answers are roughly the same length.")

# Pre-trained Model (Baseline)

In [ ]:
@dataclass
class Config:
    model_name: str = "microsoft/deberta-v3-large" 
    max_length: int = 256 
    
    epochs: int = 3
    batch_size: int = 2  
    grad_accum_steps: int = 8 # Effective batch size = 16
    learning_rate: float = 5e-6 
    weight_decay: float = 0.01
    warmup_ratio: float = 0.15
    max_grad_norm: float = 1.0
    
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "smart-mcq-solver"
    wandb_run_name: str = "deberta-large-augmented"
    
    def to_dict(self):
        return asdict(self)

cfg = Config()
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [ ]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"W&B Login Failed.")

In [ ]:
class MCQAugmentedDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row['prompt']
        choices = [row['A'], row['B'], row['C'], row['D'], row['E']]
        
        if self.is_train:
            true_label_idx = label_map[row['answer']]
            
            # Pair each choice with a boolean indicating if it's correct
            paired_choices = [(choice, i == true_label_idx) for i, choice in enumerate(choices)]
            
            # Shuffle the options dynamically!
            np.random.shuffle(paired_choices)
            
            # Unpack the shuffled choices and find the new correct index
            choices = [pair[0] for pair in paired_choices]
            item_label = next(i for i, pair in enumerate(paired_choices) if pair[1] == True)
        else:
            # During validation/testing, keep original order
            item_label = label_map.get(row.get('answer', 'A'), 0) 

        prompts = [prompt] * 5
        
        tokenized = self.tokenizer(
            prompts, choices, truncation=True, max_length=self.max_length, padding=False
        )
        
        item = {
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask']
        }
        
        if self.is_train or 'answer' in row:
            item['label'] = item_label
            
        return item

def dynamic_collate_fn(features):
    """Dynamically pads the batch to the longest sequence within the batch to save VRAM"""
    batch_size = len(features)
    num_choices = 5
    
    # Flatten the features to pad them all together
    flat_input_ids = [ids for feature in features for ids in feature['input_ids']]
    flat_attention_mask = [mask for feature in features for mask in feature['attention_mask']]
    
    # Pad using PyTorch
    max_len = max(len(x) for x in flat_input_ids)
    
    padded_input_ids = torch.zeros((batch_size * num_choices, max_len), dtype=torch.long)
    padded_attention_mask = torch.zeros((batch_size * num_choices, max_len), dtype=torch.long)
    
    for i in range(batch_size * num_choices):
        length = len(flat_input_ids[i])
        padded_input_ids[i, :length] = torch.tensor(flat_input_ids[i])
        padded_attention_mask[i, :length] = torch.tensor(flat_attention_mask[i])
        
    # Reshape back to (batch_size, num_choices, seq_len) required by AutoModelForMultipleChoice
    batch = {
        'input_ids': padded_input_ids.view(batch_size, num_choices, -1),
        'attention_mask': padded_attention_mask.view(batch_size, num_choices, -1)
    }
    
    if 'label' in features[0]:
        batch['labels'] = torch.tensor([f['label'] for f in features], dtype=torch.long)
        
    return batch

In [ ]:
# MAP@3 Metric
def compute_map3(preds, labels):
    top_3 = np.argsort(-preds, axis=1)[:, :3]
    scores = []
    for p, l in zip(top_3, labels):
        if l == p[0]: scores.append(1.0)
        elif l == p[1]: scores.append(0.5)
        elif l == p[2]: scores.append(1.0/3.0)
        else: scores.append(0.0)
    return np.mean(scores)

## Training

In [ ]:
def run_training(cfg: Config):
    run = wandb.init(project=cfg.wandb_project, name=cfg.wandb_run_name, config=cfg.to_dict(), reinit=True)
    
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    
    df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=cfg.seed, stratify=df['answer'])
    
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForMultipleChoice.from_pretrained(cfg.model_name)
    model.to(cfg.device)
    
    train_dataset = MCQAugmentedDataset(train_df, tokenizer, cfg.max_length, is_train=True)
    val_dataset = MCQAugmentedDataset(val_df, tokenizer, cfg.max_length, is_train=False) 
    
    # dynamic_collate_fn remains exactly the same as your previous code
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, collate_fn=dynamic_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, collate_fn=dynamic_collate_fn)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, eps=1e-6)
    total_steps = len(train_loader) // cfg.grad_accum_steps * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * cfg.warmup_ratio), num_training_steps=total_steps)
    
    best_map3 = 0.0
    global_step = 0
    
    for epoch in range(cfg.epochs):
        model.train()
        epoch_train_loss = 0
        
        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(cfg.device)
            attention_mask = batch['attention_mask'].to(cfg.device)
            labels = batch['labels'].to(cfg.device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / cfg.grad_accum_steps
            loss.backward()
            
            epoch_train_loss += (loss.item() * cfg.grad_accum_steps)

            wandb.log({
                "train/batch_loss": loss.item() * cfg.grad_accum_steps,
                "train/learning_rate": scheduler.get_last_lr()[0]
            }, step=global_step)
            
            if (step + 1) % cfg.grad_accum_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1
                
        # Validation Loop 
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(cfg.device)
                attention_mask = batch['attention_mask'].to(cfg.device)
                labels = batch['labels'].numpy() 
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                val_preds.append(outputs.logits.cpu().numpy())
                val_labels.extend(labels)
                
        val_preds = np.vstack(val_preds)
        val_map3 = compute_map3(val_preds, val_labels) # Ensure compute_map3 is loaded

        wandb.log({
            "epoch": epoch + 1,
            "train/epoch_loss": epoch_train_loss / len(train_loader),
            "val/map3": val_map3
        }, step=global_step)
        
        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss/len(train_loader):.4f} | Val MAP@3: {val_map3:.4f}")
        
        if val_map3 > best_map3:
            best_map3 = val_map3
            torch.save(model.state_dict(), "best_deberta_model.pth")
            
    wandb.finish()

run_training(cfg)

## Inference

In [ ]:
class InferenceConfig:
    model_name = "microsoft/deberta-v3-large" 
    model_path = "best_deberta_model.pth" 
    test_csv_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    max_length = 256
    batch_size = 8 
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

idx_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# Test Dataset & Collate Function
class MCQTestDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row['prompt']
        choices = [row['A'], row['B'], row['C'], row['D'], row['E']]
        
        prompts = [prompt] * 5
        
        tokenized = self.tokenizer(
            prompts,
            choices,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None 
        )
        
        return {
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
            'id': row['id'] # Keep track of the ID for the submission file
        }

def dynamic_collate_fn_test(features):
    batch_size = len(features)
    num_choices = 5
    
    flat_input_ids = [ids for feature in features for ids in feature['input_ids']]
    flat_attention_mask = [mask for feature in features for mask in feature['attention_mask']]
    
    max_len = max(len(x) for x in flat_input_ids)
    
    padded_input_ids = torch.zeros((batch_size * num_choices, max_len), dtype=torch.long)
    padded_attention_mask = torch.zeros((batch_size * num_choices, max_len), dtype=torch.long)
    
    for i in range(batch_size * num_choices):
        length = len(flat_input_ids[i])
        padded_input_ids[i, :length] = torch.tensor(flat_input_ids[i])
        padded_attention_mask[i, :length] = torch.tensor(flat_attention_mask[i])
        
    batch = {
        'input_ids': padded_input_ids.view(batch_size, num_choices, -1),
        'attention_mask': padded_attention_mask.view(batch_size, num_choices, -1),
        'id': [f['id'] for f in features]
    }
    
    return batch

def format_predictions(preds):
    # Get the indices of the top 3 predictions, sorted descending by logit
    top_3_indices = np.argsort(-preds, axis=1)[:, :3]
    
    # Map indices back to letters
    formatted_preds = []
    for row in top_3_indices:
        letters = [idx_to_label[idx] for idx in row]
        formatted_preds.append(" ".join(letters)) # e.g., "A B C"
        
    return formatted_preds

def generate_submission():
    print("Loading tokenizer and model architecture...")
    tokenizer = AutoTokenizer.from_pretrained(InferenceConfig.model_name)
    model = AutoModelForMultipleChoice.from_pretrained(InferenceConfig.model_name)
    
    print(f"Loading trained weights from {InferenceConfig.model_path}...")
    model.load_state_dict(torch.load(InferenceConfig.model_path, map_location=InferenceConfig.device))
    model.to(InferenceConfig.device)
    model.eval()
    
    print("Preparing test data...")
    test_df = pd.read_csv(InferenceConfig.test_csv_path)
    test_dataset = MCQTestDataset(test_df, tokenizer, InferenceConfig.max_length)
    test_loader = DataLoader(test_dataset, batch_size=InferenceConfig.batch_size, shuffle=False, collate_fn=dynamic_collate_fn_test)
    
    all_preds = []
    all_ids = []
    
    print("Running inference...")
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(InferenceConfig.device)
            attention_mask = batch['attention_mask'].to(InferenceConfig.device)
            
            # Use mixed precision for faster inference
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
                
            all_preds.append(logits.cpu().numpy())
            all_ids.extend(batch['id'])
            
    # Concatenate all batches
    all_preds = np.vstack(all_preds)
    print("Formatting predictions for MAP@3...")
    final_predictions = format_predictions(all_preds)
    
    # Create the submission DataFrame
    submission_df = pd.DataFrame({
        'id': all_ids,
        'Prediction': final_predictions
    })
    
    # Save to CSV without the Pandas index
    submission_df.to_csv('submission.csv', index=False)
    
    print("\nInference complete! Saved to 'submission.csv'.")
    print(submission_df.head())

generate_submission()